# GLADA Multi-Seed Consistency Verification

**Grouped Layer Adversarial Domain Adaptation for Audio Deepfake Detection**

Jakob Kauffmann | MS CS Thesis | San José State University | Advisor: Dr. Faranak Abri

---

## Purpose

This notebook addresses Dr. Abri's request to verify **multi-seed consistency** of GLADA results (Investigation 6). Reliable ML systems should produce consistent results across different random initializations (Picard, 2021).

### Current state
- WavLM-Large: seeds 42 and 123 already completed
- All other ablation configs: single seed (42) only

### Plan
1. Run seed 456 on three key configurations:
   - `full_glada` (all components)
   - `ablation_no_glada` (no per-group DA — backbone-only baseline comparison)
   - `ablation_no_cka` (no CKA diversity loss — tests diversity enforcement contribution)
2. Compare mean ± std across seeds for headline numbers
3. Verify that rank ordering of ablations is preserved across seeds
4. Assess statistical significance of key comparisons

### Time estimate
~2h per config on A100 (3+17 epoch reduced schedule), ~6h total.

## 0. Environment Setup

In [1]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

# Copy and install glada package from Drive
import subprocess, sys, shutil, os
DRIVE_PKG = '/content/drive/MyDrive/Jakob/glada_pkg'
LOCAL_PKG = '/content/glada_pkg'
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', LOCAL_PKG])
if LOCAL_PKG not in sys.path:
    sys.path.insert(0, LOCAL_PKG)
print(f'Installed glada package from {LOCAL_PKG}')

# GPU check
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_mem_gb = props.total_memory / 1e9
    print(f"GPU: {props.name}, {total_mem_gb:.1f} GB")

    # Enable TF32 for A100 (2x throughput on matmuls)
    if 'A100' in props.name or total_mem_gb > 30:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("  TF32 enabled (A100 detected)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

Mounted at /content/drive
Installed glada package from /content/glada_pkg
GPU: NVIDIA A100-SXM4-80GB, 85.1 GB
  TF32 enabled (A100 detected)


## 1. Dataset Extraction

Datasets are stored as compressed archives on Google Drive. To avoid filling the Colab runtime SSD (~80 GB), we extract one archive at a time: copy → extract → delete archive.

**Required datasets for training:**
- ASVspoof 2019 LA (train + eval): Source domain with labeled spoof/bonafide samples
- In-The-Wild (ITW): Target domain (unlabeled for DA) + evaluation set

In [ ]:
import os
import shutil
import tarfile
import zipfile

DRIVE_DATASETS = '/content/drive/MyDrive/Jakob/Datasets'
LOCAL_DATA = '/content/data'


def extract_one_dataset(drive_archive, extract_to, verify_path, name):
    """Copy archive from Drive -> runtime SSD, extract, delete archive.

    Processes one dataset at a time so the runtime SSD never holds
    more than one compressed archive alongside its decompressed data.
    """
    if os.path.isdir(verify_path) and os.listdir(verify_path):
        print(f'  [{name}] Already extracted at {verify_path}')
        return

    archive_name = os.path.basename(drive_archive)
    local_archive = f'/content/{archive_name}'

    # Step 1: Copy archive to runtime SSD
    print(f'  [{name}] Copying {archive_name} to local SSD...')
    shutil.copy2(drive_archive, local_archive)

    # Step 2: Extract
    print(f'  [{name}] Extracting...')
    os.makedirs(extract_to, exist_ok=True)
    if archive_name.endswith(('.tar.gz', '.tgz', '.tar')):
        mode = 'r:gz' if archive_name.endswith(('.tar.gz', '.tgz')) else 'r:'
        with tarfile.open(local_archive, mode) as tar:
            tar.extractall(extract_to)
    elif archive_name.endswith('.zip'):
        with zipfile.ZipFile(local_archive, 'r') as zf:
            zf.extractall(extract_to)
    else:
        raise ValueError(f'Unknown archive format: {archive_name}')

    # Step 3: Delete local archive (free SSD space)
    os.remove(local_archive)
    print(f'  [{name}] Done. Verified at {verify_path}')


# --- ASVspoof 2019 LA ---
extract_one_dataset(
    drive_archive=os.path.join(DRIVE_DATASETS, 'LA.tar'),
    extract_to=os.path.join(LOCAL_DATA, 'asvspoof'),
    verify_path=os.path.join(LOCAL_DATA, 'asvspoof', 'LA'),
    name='ASVspoof 2019 LA',
)

# --- In-The-Wild ---
extract_one_dataset(
    drive_archive=os.path.join(DRIVE_DATASETS, 'release_in_the_wild.zip'),
    extract_to=os.path.join(LOCAL_DATA, 'itw'),
    verify_path=os.path.join(LOCAL_DATA, 'itw', 'release_in_the_wild'),
    name='In-The-Wild',
)

# Verify paths
ASVSPOOF_ROOT = os.path.join(LOCAL_DATA, 'asvspoof', 'LA')
ITW_ROOT = os.path.join(LOCAL_DATA, 'itw', 'release_in_the_wild')

for path, name in [(ASVSPOOF_ROOT, 'ASVspoof'), (ITW_ROOT, 'ITW')]:
    exists = os.path.isdir(path)
    print(f"  {name}: {'OK' if exists else 'MISSING'} {path}")

  [ASVspoof 2019 LA] Copying LA.tar to local SSD...
  [ASVspoof 2019 LA] Extracting...


/tmp/ipykernel_815/317364980.py:33: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_to)


  [ASVspoof 2019 LA] Done. Verified at /content/data/asvspoof/LA
  [In-The-Wild] Copying release_in_the_wild.zip to local SSD...
  [In-The-Wild] Extracting...


## 2. Imports and Configuration

In [ ]:
import json
import time
import numpy as np
import torch
import torch.nn as nn

from glada import GLADAConfig, set_seed
from glada.models import GLADAModel
from glada.trainers.glada_trainer import GLADATrainer
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)

print("All imports successful.")

## 3. Define Multi-Seed Configurations

We run three key configurations at seed 456 to complement existing seed 42 results:

1. **full_glada**: All GLADA components enabled — the primary model.
2. **ablation_no_glada**: Per-group DA disabled — tests whether the novel contribution helps.
3. **ablation_no_cka**: CKA diversity loss disabled — tests whether group diversity enforcement matters.

These three configurations span the most important comparison axis for the thesis: full system vs. the system without its primary contribution (GLADA) vs. a key supporting component (CKA).

### Reduced epoch budget

Following the ablation study protocol, we use a reduced epoch budget (3 + 17 = 20 epochs) rather than the full 5 + 45 = 50 epochs. This is sufficient for relative comparison and was used for all existing ablation results reported in the thesis.

In [ ]:
# ============================================================================
# Multi-seed configurations (seed 456)
# Reduced epoch budget (3+17) to match ablation study protocol.
# All other hyperparameters match the ablation study defaults.
# ============================================================================

SEED = 456
BACKBONE = 'wavlm-base-plus'
PHASE1 = 3   # Reduced from 5 (ablation protocol)
PHASE2 = 17  # Reduced from 45 (ablation protocol)

# Detect GPU type for batch size and AMP dtype
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    AMP_DTYPE = 'bfloat16' if total_mem_gb > 30 else 'float16'
    BATCH_SIZE = 24  # Conservative; works on both T4 and A100
    print(f"GPU: {torch.cuda.get_device_name(0)}, {total_mem_gb:.1f} GB")
    print(f"  AMP dtype: {AMP_DTYPE}, Batch size: {BATCH_SIZE}")
else:
    AMP_DTYPE = 'float16'
    BATCH_SIZE = 4

# --- Config 1: Full GLADA (all components) ---
config_full = GLADAConfig(
    backbone=BACKBONE,
    seed=SEED,
    experiment_name=f'multiseed_full_glada_s{SEED}',
    phase1_epochs=PHASE1,
    phase2_epochs=PHASE2,
    batch_size=BATCH_SIZE,
    amp_dtype=AMP_DTYPE,
    num_workers=0,  # Always 0 on Colab
)

# --- Config 2: No GLADA (no per-group DA) ---
config_no_glada = GLADAConfig(
    backbone=BACKBONE,
    seed=SEED,
    experiment_name=f'multiseed_no_glada_s{SEED}',
    phase1_epochs=PHASE1,
    phase2_epochs=PHASE2,
    batch_size=BATCH_SIZE,
    amp_dtype=AMP_DTYPE,
    num_workers=0,
    use_glada=False,  # Disable per-group DA
)

# --- Config 3: No CKA (no diversity loss) ---
config_no_cka = GLADAConfig(
    backbone=BACKBONE,
    seed=SEED,
    experiment_name=f'multiseed_no_cka_s{SEED}',
    phase1_epochs=PHASE1,
    phase2_epochs=PHASE2,
    batch_size=BATCH_SIZE,
    amp_dtype=AMP_DTYPE,
    num_workers=0,
    use_cka_loss=False,  # Disable CKA diversity loss
)

CONFIGS = [config_full, config_no_glada, config_no_cka]

print(f"\nConfigurations to run (seed={SEED}, {PHASE1}+{PHASE2}={PHASE1+PHASE2} epochs):")
for i, c in enumerate(CONFIGS, 1):
    print(f"  {i}. {c.experiment_name}")
    print(f"     glada={c.use_glada}, vib={c.use_vib}, spk={c.use_speaker_adversary}, "
          f"cka={c.use_cka_loss}, hier={c.use_hierarchical}, rb={c.use_rawboost}")

GPU: NVIDIA A100-SXM4-80GB, 85.1 GB
  AMP dtype: bfloat16, Batch size: 24

Configurations to run (seed=456, 3+17=20 epochs):
  1. multiseed_full_glada_s456
     glada=True, vib=True, spk=True, cka=True, hier=True, rb=True
  2. multiseed_no_glada_s456
     glada=False, vib=True, spk=True, cka=True, hier=True, rb=True
  3. multiseed_no_cka_s456
     glada=True, vib=True, spk=True, cka=False, hier=True, rb=True


## 4. Load Datasets

Datasets are loaded once and reused across all configurations. This avoids redundant I/O and ensures identical data across seeds (only the random initialization differs).

In [ ]:
# ============================================================================
# Load datasets ONCE, reuse across all configs.
# Source: ASVspoof 2019 LA training set (labeled, with speaker IDs)
# Target: In-The-Wild (unlabeled for domain adaptation)
# Eval: In-The-Wild evaluation split (labeled, primary metric)
# Extra eval: ASVspoof 2019 LA eval set (in-domain sanity check)
#
# All dataset constructors take a GLADAConfig object (not individual kwargs).
# We use config_full for ALL datasets (including eval). The same max_audio_len
# (64000 = 4s at 16kHz) is applied to eval to prevent zero-length audio from
# crashing the CNN feature extractor.
# ============================================================================

# Paths (must match Colab extraction layout)
PROTO_TRAIN = os.path.join(ASVSPOOF_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.train.trn.txt')
AUDIO_TRAIN = os.path.join(ASVSPOOF_ROOT, 'ASVspoof2019_LA_train', 'flac')
PROTO_EVAL = os.path.join(ASVSPOOF_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.eval.trl.txt')
AUDIO_EVAL = os.path.join(ASVSPOOF_ROOT, 'ASVspoof2019_LA_eval', 'flac')
ITW_META = os.path.join(ITW_ROOT, 'meta.csv')
ITW_AUDIO = ITW_ROOT

print("Loading datasets...")

# Source domain: ASVspoof 2019 LA training set
source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=PROTO_TRAIN,
    audio_dir=AUDIO_TRAIN,
    config=config_full,
    augmenter=None,
)
print(f"  Source (ASVspoof train): {len(source_ds)} samples, {len(source_ds.speaker_to_id)} speakers")

# Target domain: ITW (unlabeled for DA)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config_full,
    use_labels=False,
)
print(f"  Target (ITW train): {len(target_ds)} samples")

# Evaluation: ITW with ground truth labels for EER computation.
# Uses same config (max_audio_len=64000) — NOT 0. Zero-length audio crashes
# the CNN feature extractor ("Kernel size can't be greater than input size").
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config_full,
    use_labels=True,
)
print(f"  Eval (ITW eval): {len(eval_ds)} samples")

# Extra evaluation: ASVspoof 2019 LA eval set (in-domain)
la_eval_ds = ASVspoofEvalDataset(
    protocol_path=PROTO_EVAL,
    audio_dir=AUDIO_EVAL,
    config=config_full,
)
print(f"  Extra eval (ASVspoof 2019 LA eval): {len(la_eval_ds)} samples")

extra_eval = {'2019_LA': la_eval_ds}
NUM_SPEAKERS = len(source_ds.speaker_to_id)
print(f"\nDatasets loaded. {NUM_SPEAKERS} speakers detected.")

Loading datasets...
  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  Source (ASVspoof train): 25380 samples, 20 speakers
  ITW: 31779 samples (unlabeled returned)
  Target (ITW train): 31779 samples
  ITW: 31779 samples (labeled returned)
  Eval (ITW eval): 31779 samples
  ASVspoof Eval: 71237 samples
  Extra eval (ASVspoof 2019 LA eval): 71237 samples

Datasets loaded. 20 speakers detected.


## 5. Training Loop

Runs each configuration sequentially. Between configs:
- GPU memory is cleared
- Seed is re-set for reproducibility
- Results are saved incrementally to Drive (survives Colab disconnections)

### Resume support

If Colab disconnects mid-run, re-run all cells from the top. Already-completed experiments (detected by existing `history.json`) will be automatically skipped.

In [ ]:
# ============================================================================
# Sequential training of all configs.
# Skips configs that already have results on Drive.
# Saves results incrementally for Colab resilience.
# ============================================================================
import gc

all_run_results = {}
total_start = time.time()

for idx, config in enumerate(CONFIGS, 1):
    print(f"\n{'=' * 60}")
    print(f"CONFIG {idx}/{len(CONFIGS)}: {config.experiment_name}")
    print(f"{'=' * 60}")

    # Check if already completed (resume support)
    import glob
    existing = glob.glob(os.path.join(config.drive_log_dir, config.experiment_name, '*', 'history.json'))
    if existing:
        print(f"  SKIPPING -- already completed. Found: {existing[0]}")
        # Load existing results
        with open(existing[0]) as f:
            hist = json.load(f)
        best_eer = min(
            h.get('eval', {}).get('ITW', {}).get('eer', 999)
            for h in hist
        )
        all_run_results[config.experiment_name] = {
            'best_eer': best_eer,
            'epochs': len(hist),
            'status': 'loaded_from_disk',
        }
        continue

    # Set seed for this config
    set_seed(config.seed)

    # Create DataLoaders (re-created per config for batch_size flexibility)
    loaders = create_dataloaders(config, source_ds, target_ds, eval_ds, extra_eval)

    try:
        trainer = GLADATrainer(
            config=config,
            source_loader=loaders['source'],
            target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items()
                                if k not in ('source', 'target', 'eval')},
            num_speakers=NUM_SPEAKERS,
        )
        results = trainer.train()
        all_run_results[config.experiment_name] = {
            'best_eer': results['best_eer'],
            'best_epoch': results['best_epoch'],
            'run_dir': results['run_dir'],
            'status': 'completed',
        }
        print(f"\n  Completed: Best ITW EER = {results['best_eer']:.2f}% (epoch {results['best_epoch']})")

    except Exception as e:
        print(f"\n  FAILED: {e}")
        all_run_results[config.experiment_name] = {'error': str(e), 'status': 'failed'}

    # GPU cleanup between configs
    if 'trainer' in dir():
        del trainer
    torch.cuda.empty_cache()
    gc.collect()

    # Save incremental results to Drive
    incremental_path = os.path.join(config.drive_log_dir, 'multiseed_results.json')
    with open(incremental_path, 'w') as f:
        json.dump(all_run_results, f, indent=2, default=str)
    print(f"  Incremental results saved to {incremental_path}")

total_time = (time.time() - total_start) / 60
print(f"\n{'=' * 60}")
print(f"All configs complete in {total_time:.1f} minutes.")
print(f"{'=' * 60}")


CONFIG 1/3: multiseed_full_glada_s456
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/multiseed_full_glada_s456/20260324_175636

GLADA Training | wavlm-base-plus | seed=456
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.001, SPK=0.1, CKA=0.01


Ep 1:   0%|          | 0/1057 [00:00<?, ?it/s]

... (275 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  [WARN] KL clamped 17000x consecutive (raw=2.099e+04, clamp=10.0)
  [ADAPT] VIB beta halved: 3.0517578125e-08 -> 1.52587890625e-08 (KL raw avg=8352.2 > 1000)


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  Ep 20 P2: L=0.337 KL=10.00 | ITW=21.73% | 2019_LA=1.75%

Done in 149.8 min | Best ITW EER: 16.15% (Ep 7)

  Completed: Best ITW EER = 16.15% (epoch 7)
  Incremental results saved to /content/drive/MyDrive/Jakob/Experiments/multiseed_results.json

All configs complete in 427.8 minutes.


---

## 6. Multi-Seed Analysis

Now we aggregate results across all available seeds (42, 123, 456) and compute mean ± standard deviation to assess consistency.

### Statistical context

- **2 seeds** (current for WavLM-Large): Provides a range but no meaningful variance estimate.
- **3 seeds** (target): Allows mean ± std, which is the minimum for reporting in ML publications (Picard, 2021).
- **Rank consistency**: If the rank ordering of ablation configs is preserved across seeds, the relative comparisons are robust even if absolute numbers vary.

In [ ]:
# ============================================================================
# Multi-Seed Analysis
# Aggregates results from all available seeds across key configurations.
# ============================================================================

import glob
import json
import numpy as np
import matplotlib.pyplot as plt

EXP_DIR = '/content/drive/MyDrive/Jakob/Experiments'

# --- Load comprehensive results ---
results_path = os.path.join(EXP_DIR, 'comprehensive_eval', 'results.json')
comp_results = {}
if os.path.exists(results_path):
    with open(results_path) as f:
        comp_results = json.load(f)
    print(f"Loaded comprehensive results: {len(comp_results)} models")

# --- Also scan individual experiment directories for history.json ---
def get_best_itw_eer(experiment_name):
    """Get best ITW EER from an experiment's history.json."""
    # Check comprehensive results first
    for key, data in comp_results.items():
        if experiment_name.lower() in key.lower():
            itw = data.get('ITW', {})
            if 'eer' in itw:
                return itw['eer']

    # Fallback to history.json
    pattern = os.path.join(EXP_DIR, experiment_name, '*', 'history.json')
    matches = glob.glob(pattern)
    if not matches:
        pattern = os.path.join(EXP_DIR, experiment_name, 'history.json')
        matches = glob.glob(pattern)
    if not matches:
        return None

    with open(sorted(matches)[-1]) as f:
        hist = json.load(f)
    eers = [h.get('eval', {}).get('ITW', {}).get('eer', 999) for h in hist]
    return min(eers) if eers else None



# Map config types to same-config, different-seed experiment names ONLY.
# Do NOT mix different backbones — that's a backbone comparison, not multi-seed.
CONFIG_GROUPS = {
    'Full GLADA (WavLM-Large)': [
        'ablation_full_wavlm-large',
        'ablation_full_wavlm-large_s123',
        'ablation_full_wavlm-large_s2026',
    ],
    'Full GLADA (WavLM-BP)': [
        'ablation_full_glada',
        'multiseed_full_glada_s456',
    ],
    'No GLADA': [
        'ablation_no_glada',
        'multiseed_no_glada_s456',
    ],
    'Global DA Only': [
        'ablation_global_da_only',
    ],
    'No DA (backbone only)': [
        'ablation_backbone_only',
        'ablation_no_da',
    ],
    'No CKA': [
        'ablation_no_cka',
        'multiseed_no_cka_s456',
    ],
    'No VIB': ['ablation_no_vib'],
    'No Speaker': ['ablation_no_speaker'],
    'No Hierarchical': ['ablation_no_hier'],
    'No RawBoost': ['ablation_no_rawboost'],
}

print("=" * 80)
print("INVESTIGATION 6: Multi-Seed Consistency")
print("=" * 80)
print()
print(f"{'Configuration':<30s} {'Seeds':>6s} {'Mean EER':>10s} {'Std':>8s} {'Values'}")
print("-" * 90)

config_means = {}
for label, name_variants in CONFIG_GROUPS.items():
    eers = []
    for name in name_variants:
        eer = get_best_itw_eer(name)
        if eer is not None and eer < 100:
            eers.append(eer)

    # Also scan comprehensive results for any matching keys
    for key, data in comp_results.items():
        for name in name_variants:
            if name.lower() in key.lower() and key not in name_variants:
                itw = data.get('ITW', {})
                if 'eer' in itw and itw['eer'] < 100:
                    eer_val = itw['eer']
                    if eer_val not in eers:  # Avoid duplicates
                        eers.append(eer_val)

    eers = sorted(set(eers))  # Deduplicate
    config_means[label] = eers

    if len(eers) == 0:
        print(f"  {label:<28s} {'0':>6s} {'N/A':>10s} {'N/A':>8s}")
    elif len(eers) == 1:
        print(f"  {label:<28s} {'1':>6s} {eers[0]:>9.2f}% {'N/A':>8s} {eers}")
    else:
        mean = np.mean(eers)
        std = np.std(eers)
        print(f"  {label:<28s} {len(eers):>6d} {mean:>9.2f}% {std:>7.2f}% {[f'{e:.2f}' for e in eers]}")

In [ ]:
# ============================================================================
# Rank Consistency Analysis
# Checks whether the rank ordering of configs is preserved across seeds.
# This is more important than absolute numbers -- even if variance is high,
# consistent rankings mean relative comparisons are reliable.
# ============================================================================

print("\n" + "=" * 80)
print("INVESTIGATION 6b: Rank Ordering Consistency")
print("=" * 80)
print()

# For configs with at least 2 seeds, check ranking
multi_seed_configs = {k: v for k, v in config_means.items() if len(v) >= 2}

if len(multi_seed_configs) < 2:
    print("Not enough multi-seed configs for rank comparison.")
    print("After running seed 456 experiments, re-run this cell.")
else:
    # Compare mean EERs
    ranked = sorted(multi_seed_configs.items(), key=lambda x: np.mean(x[1]))
    print("Ranking by mean ITW EER (lower is better):")
    for rank, (label, eers) in enumerate(ranked, 1):
        mean = np.mean(eers)
        std = np.std(eers)
        print(f"  {rank}. {label:<30s} {mean:.2f} +/- {std:.2f}%")

    # Check if Full GLADA ranks #1 (validates the thesis contribution)
    print()
    if ranked[0][0] == 'Full GLADA':
        print("[OK] Full GLADA ranks #1 -- the primary contribution shows consistent benefit.")
    else:
        print(f"[NOTE] Full GLADA does not rank #1 (ranked by: {ranked[0][0]}). Investigate further.")

    # Check overlap in confidence intervals
    if len(multi_seed_configs) >= 2:
        print("\nPairwise comparisons (non-overlapping ranges suggest significant difference):")
        items = list(multi_seed_configs.items())
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                name_i, eers_i = items[i]
                name_j, eers_j = items[j]
                mean_i, mean_j = np.mean(eers_i), np.mean(eers_j)
                diff = abs(mean_i - mean_j)
                range_i = max(eers_i) - min(eers_i)
                range_j = max(eers_j) - min(eers_j)
                overlap = max(0, min(max(eers_i), max(eers_j)) - max(min(eers_i), min(eers_j)))
                print(f"  {name_i} vs {name_j}:")
                print(f"    Delta mean = {diff:.2f}%, Range overlap = {'YES' if overlap > 0 else 'NO'}")


INVESTIGATION 6b: Rank Ordering Consistency

Ranking by mean ITW EER (lower is better):
  1. No DA (backbone only)          13.85 +/- 0.45%
  2. No CKA                         14.44 +/- 1.71%
  3. Full GLADA (WavLM-Large)       14.45 +/- 1.40%
  4. No GLADA                       17.40 +/- 2.38%
  5. Full GLADA (WavLM-BP)          19.78 +/- 0.14%

[NOTE] Full GLADA does not rank #1 (ranked by: No DA (backbone only)). Investigate further.

Pairwise comparisons (non-overlapping ranges suggest significant difference):
  Full GLADA (WavLM-Large) vs Full GLADA (WavLM-BP):
    Delta mean = 5.33%, Range overlap = NO
  Full GLADA (WavLM-Large) vs No GLADA:
    Delta mean = 2.95%, Range overlap = YES
  Full GLADA (WavLM-Large) vs No DA (backbone only):
    Delta mean = 0.60%, Range overlap = YES
  Full GLADA (WavLM-Large) vs No CKA:
    Delta mean = 0.01%, Range overlap = YES
  Full GLADA (WavLM-BP) vs No GLADA:
    Delta mean = 2.39%, Range overlap = YES
  Full GLADA (WavLM-BP) vs No DA (backb

In [ ]:
# ============================================================================
# Multi-Seed Visualization
# Bar chart with error bars showing mean +/- std for each configuration.
# ============================================================================

# Only plot configs that have results
plot_configs = {k: v for k, v in config_means.items() if len(v) >= 1}

if plot_configs:
    fig, ax = plt.subplots(figsize=(12, 6))

    labels = list(plot_configs.keys())
    means = [np.mean(v) for v in plot_configs.values()]
    stds = [np.std(v) if len(v) > 1 else 0 for v in plot_configs.values()]
    n_seeds = [len(v) for v in plot_configs.values()]

    colors = ['#2196F3' if 'Full' in l else '#FF9800' for l in labels]

    bars = ax.bar(range(len(labels)), means, yerr=stds, capsize=8,
                  color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

    # Annotate with seed counts
    for i, (bar, n, m, s) in enumerate(zip(bars, n_seeds, means, stds)):
        text = f'{m:.2f}%'
        if s > 0:
            text += f'\n+/-{s:.2f}'
        text += f'\n(n={n})'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(stds) * 0.3,
                text, ha='center', va='bottom', fontsize=9)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('ITW EER (%)', fontsize=12)
    ax.set_title('Multi-Seed Results: Mean +/- Std ITW EER', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    save_path = os.path.join(EXP_DIR, 'diagnostics', 'investigation6_multiseed.png')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
else:
    print("No results available yet. Run training first (Section 5).")

---

## Summary

| Configuration | Seeds | Mean ITW EER | Std | Consistent? |
|---|---|---|---|---|
| Full GLADA | _[fill]_ | _[fill]_ | _[fill]_ | _[fill]_ |
| No GLADA | _[fill]_ | _[fill]_ | _[fill]_ | _[fill]_ |
| No CKA | _[fill]_ | _[fill]_ | _[fill]_ | _[fill]_ |

### Key findings (fill after running)

1. **Absolute consistency**: Are the standard deviations across seeds reasonable (< 2% EER)?
2. **Rank consistency**: Is Full GLADA consistently better than No GLADA across seeds?
3. **Statistical significance**: Is the gap between Full GLADA and No GLADA larger than the within-config variance?

---

## References

1. **Picard, D.** "Torch.manual_seed(3407) is all you need: On the influence of random seeds in deep learning architectures for computer vision", 2021 -- Motivation for multi-seed evaluation
2. **Ganin et al.** "Domain-Adversarial Training of Neural Networks", JMLR 2016 -- GRL framework
3. **Kornblith et al.** "Similarity of Neural Network Representations Revisited", ICML 2019 -- CKA metric
4. **Huang et al.** "IB-CAAN", 2025 -- Baseline comparison